# LDA (hitungan kata)
**text › topic_modeling** · notebook `lda`

**Kapan dipakai:** Model probabilistik klasik: tiap dokumen campuran topik. Cocok bila perlu distribusi topik per dokumen.

**Jebakan umum:** Lambat dan peka terhadap hiperparameter dan stopword; topik sering kurang tajam dibanding NMF pada teks pendek.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/text/classification_en/news4_train.csv"
TEST_PATH  = ROOT / "data_latihan/text/classification_en/news4_test.csv"
TEXT_COL   = "text"
LABEL_COL  = "label"            # hanya untuk EVALUASI; algoritma tidak melihatnya
LANG       = "en"
USE_STOPWORDS = True
SEED = 42
MODEL_KEY  = "lda"
OUT = out_dir("text", "topic_modeling", MODEL_KEY)
set_seed(SEED)
from superdb.text import ID_STOPWORDS
STOP = (ID_STOPWORDS if LANG == "id" else "english") if USE_STOPWORDS else None
K = 4                      # jumlah topik (tentukan dari sapuan K di bawah)

## 2. Baca data
Label kategori **tidak dipakai** saat melatih; hanya untuk mengecek seberapa selaras topik dengan kategori asli.

In [ ]:
from superdb.text import clean_text
df = pd.read_csv(TRAIN_PATH); texts = df[TEXT_COL].map(clean_text).tolist(); true = df[LABEL_COL].to_numpy()
print(len(texts), "dokumen;", len(set(true)), "kategori asli:", sorted(set(true)))

## 3. Latih model topik

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vec = CountVectorizer(min_df=3, max_df=0.5, stop_words=STOP)
model = LatentDirichletAllocation(n_components=K, learning_method="batch", max_iter=30, random_state=42)
Xv = vec.fit_transform(texts)
W = model.fit_transform(Xv)                        # bobot topik per dokumen
terms = np.asarray(vec.get_feature_names_out())
H = model.components_
topics = [list(terms[np.argsort(-H[k])[:10]]) for k in range(K)]
for k, t in enumerate(topics):
    print(f"topik {k}: {', '.join(t)}")

## 4. Seberapa baik topik? (tiga cara)
1. **NMI/ARI/purity** terhadap kategori asli (hanya mungkin karena data ini berlabel).
2. **Koherensi UMass** (tanpa label): makin mendekati 0 makin masuk akal.
3. **Dibaca manusia**: apakah daftar kata membentuk satu tema? Ini yang akhirnya dinilai juri.

In [ ]:
from sklearn.metrics import normalized_mutual_info_score as nmi, adjusted_rand_score as ari
from superdb.text_extra import umass_coherence, purity
assign = np.abs(W).argmax(1)
vocab = {w: i for i, w in enumerate(terms)}
coh = umass_coherence(topics, (Xv > 0), vocab)
metrics = {"NMI": nmi(true, assign), "ARI": ari(true, assign), "purity": purity(true, assign), "coherence_umass_mean": float(np.mean(coh))}
display(show_table(metrics))
display(pd.crosstab(pd.Series(assign, name="topik"), pd.Series(true, name="kategori asli")))

## 5. Memilih K: sapuan jumlah topik
Cari K di mana koherensi/NMI berhenti membaik, bukan sekadar maksimum.

In [ ]:
from sklearn.base import clone
rows = []
for k in range(2, 9):
    m_ = clone(model).set_params(**{"n_components": k}); W_ = m_.fit_transform(Xv); H_ = m_.components_
    tp = [list(terms[np.argsort(-H_[i])[:10]]) for i in range(k)]
    rows.append({"K": k, "NMI": nmi(true, np.abs(W_).argmax(1)), "koherensi": float(np.mean(umass_coherence(tp, (Xv > 0), vocab)))})
sweep = pd.DataFrame(rows).set_index("K"); display(sweep.round(3))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2)); sweep.NMI.plot(marker="o", ax=ax[0], title="NMI vs K"); sweep.koherensi.plot(marker="o", ax=ax[1], title="Koherensi UMass vs K")
plt.tight_layout(); plt.savefig(OUT / "sweep_k.png", dpi=130); plt.show()

## 6. Dokumen representatif tiap topik

In [ ]:
for k in range(K):
    i = int(np.argsort(-np.abs(W[:, k]))[0]); print(f"[topik {k}] ({true[i]}) {texts[i][:200]}...")
pd.DataFrame({f"topik_{k}": t for k, t in enumerate(topics)}).to_csv(OUT / "topics.csv", index=False)
save_json({"K": K, "topics": topics, "metrics": metrics, "sweep": sweep.reset_index().to_dict("records")}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Beri **nama** tiap topik dari kata teratasnya untuk laporan.
- Topik jelek? Ubah stopword, `max_df`, atau K; gabungkan topik tumpang tindih.
- Untuk teks pendek/slang, clustering dengan embedding (folder `clustering`, bagian heavy) sering lebih rapi.